# Creating the Streamlit App — Churn Prediction

### 📌 Recap

From [04 Model Loading.ipynb](04%20Model%20Loading.ipynb): loading the trained model + encoders/scaler from disk and predicting on one new customer row was fully worked out. `app.py` (already written, in the project root) wraps that exact same logic in a **Streamlit** web UI — input widgets instead of a hardcoded dict, live prediction on every interaction.

This notebook **documents and verifies `app.py` as it actually exists** — it doesn't recreate or modify it. Streamlit apps can't be run as notebook cells (`st.*` calls need a live `streamlit run` server, not a Jupyter kernel), so the app's code is shown here as read-only reference, while the underlying data logic is independently re-verified with real, executed code against your actual project files (`model.keras`, the three `.pkl` files) and your actual project `venv` (TensorFlow 2.15.0 / Keras 2.15.0) — registered as this notebook's kernel.

## 1. Why Streamlit

Streamlit turns a plain Python script into a web app — each `st.*` call renders a widget or piece of output, with no HTML/CSS/JS required. The script re-runs top-to-bottom on every user interaction (moving a slider, picking a dropdown value), and Streamlit only updates what changed.

## 2. Imports and loading the trained artifacts

```python
import streamlit as st
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pandas as pd
import pickle

# Load the trained model
model = tf.keras.models.load_model('model.keras')

# Load the encoders and scaler
with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)
```

This mirrors [04 Model Loading.ipynb](04%20Model%20Loading.ipynb) exactly — model first, then the three pickled preprocessing artifacts, all loaded once, at import time, before any UI is drawn.

> 📝 **Version-specific note carried over from notebook 04:** loading `model.keras` there needed `compile=False` to work around a Keras **3.15** deserialization bug. Your actual project `venv` pins **TensorFlow 2.15.0 / Keras 2.15.0** — an older, pre-Keras-3 codepath — and `load_model('model.keras')` (no extra arguments) was re-tested directly against your real `model.keras` file in that exact environment below: it loads cleanly, no bug. Worth remembering only if this project's TensorFlow version is ever upgraded to 2.16+ in the future.

In [1]:
import tensorflow as tf

model = tf.keras.models.load_model("model.keras")
model.summary()

Model: "sequential"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 dense (Dense)               (None, 64)                832       


 dense_1 (Dense)             (None, 32)                2080      


 dense_2 (Dense)             (None, 1)                 33        


Total params: 2945 (11.50 KB)


Trainable params: 2945 (11.50 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


## 3. Building the input form

```python
st.title('Customer Churn PRediction')

geography = st.selectbox('Geography', onehot_encoder_geo.categories_[0])
gender = st.selectbox('Gender', label_encoder_gender.classes_)
age = st.slider('Age', 18, 92)
balance = st.number_input('Balance')
credit_score = st.number_input('Credit Score')
estimated_salary = st.number_input('Estimated Salary')
tenure = st.slider('Tenure', 0, 10)
num_of_products = st.slider('Number of Products', 1, 4)
has_cr_card = st.selectbox('Has Credit Card', [0, 1])
is_active_member = st.selectbox('Is Active Member', [0, 1])
```

| Field | Widget | Notes |
|---|---|---|
| `Geography` | dropdown | populated directly from `onehot_encoder_geo.categories_[0]` — the exact categories the encoder was fit on, so the UI can never offer a value the model wasn't trained to handle |
| `Gender` | dropdown | same idea, from `label_encoder_gender.classes_` |
| `Age`, `Tenure`, `NumOfProducts` | sliders | bounded to sensible/known ranges (`Tenure` 0–10, `NumOfProducts` 1–4, matching the dataset's real ranges) |
| `Balance`, `CreditScore`, `EstimatedSalary` | free number input | continuous values, no natural slider bounds |
| `HasCrCard`, `IsActiveMember` | dropdown of `[0, 1]` | binary flags |

> 📝 **Small mismatch between the video's narration and the actual code, worth knowing about (not a bug — just noting for accuracy):** the video's spoken description says a slider is used for `HasCrCard`/`IsActiveMember`, but the actual `app.py` uses `st.selectbox('...', [0, 1])` for both. The code is internally consistent and works fine either way — just flagging the discrepancy since this notebook is documenting the real file, not the narration.

## 4. Assembling and encoding the input row

```python
input_data = pd.DataFrame({
    'CreditScore': [credit_score],
    'Gender': [label_encoder_gender.transform([gender])[0]],
    'Age': [age],
    'Tenure': [tenure],
    'Balance': [balance],
    'NumOfProducts': [num_of_products],
    'HasCrCard': [has_cr_card],
    'IsActiveMember': [is_active_member],
    'EstimatedSalary': [estimated_salary]
})

# One-hot encode 'Geography'
geo_encoded = onehot_encoder_geo.transform([[geography]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

# Combine one-hot encoded columns with input data
input_data = pd.concat([input_data.reset_index(drop=True), geo_encoded_df], axis=1)
```

Same structure as [04 Model Loading.ipynb](04%20Model%20Loading.ipynb): build the raw fields into a DataFrame, label-encode `Gender` inline while building it, one-hot encode `Geography` separately, then concatenate.

## 5. ⚠️ Critical bug found: `.toarray()` will crash with your *current* `onehot_encoder_geo.pkl`

This isn't a hypothetical — verified directly against your actual project files below. `.toarray()` only exists on **sparse** matrices. Whether `onehot_encoder_geo.transform(...)` returns a sparse matrix or a plain dense NumPy array depends entirely on the `sparse_output` setting the encoder was **fit** with — and that choice is now baked permanently into the pickle file, regardless of what `app.py` assumes.

In [2]:
import pickle

with open("onehot_encoder_geo.pkl", "rb") as file:
    onehot_encoder_geo = pickle.load(file)

print("sparse_output on this encoder:", onehot_encoder_geo.sparse_output)

result = onehot_encoder_geo.transform([["France"]])
print("type of .transform() output:", type(result))

try:
    result.toarray()
    print(".toarray() succeeded")
except AttributeError as e:
    print(f".toarray() FAILS: {e}")

sparse_output on this encoder: False
type of .transform() output: <class 'numpy.ndarray'>
.toarray() FAILS: 'numpy.ndarray' object has no attribute 'toarray'


C:\AI Projects\ML&DL\Learning\NLP Krish\Projects\ANN Classification\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


**Confirmed:** your current `onehot_encoder_geo.pkl` was fit with `sparse_output=False`, so `.transform()` already returns a dense `numpy.ndarray` directly — and calling `.toarray()` on that (as `app.py` line 51 does) raises `AttributeError: 'numpy.ndarray' object has no attribute 'toarray'`. **If you run `streamlit run app.py` right now, it will crash as soon as you interact with the form.**

Two independent, equally valid fixes — pick whichever's more convenient, this notebook doesn't apply either automatically since you asked me not to touch `app.py`:

1. **Fix the pickle, not the code:** re-fit and re-save `onehot_encoder_geo` with the *default* setting (`OneHotEncoder()`, `sparse_output=True`) so it matches what `app.py` expects.
2. **Fix the code, not the pickle:** in `app.py`, drop the `.toarray()` call — `onehot_encoder_geo.transform([[geography]])` already returns a dense array with the encoder as currently pickled.

Want me to apply either fix for you? Just say which.

## 6. The rest of the pipeline — verified working end-to-end

Bypassing just the broken `.toarray()` call (using the encoder's actual dense output directly), the remaining logic — concatenation, column order, scaling, prediction — was re-run for real against your actual `model.keras`, `scaler.pkl`, and `label_encoder_gender.pkl`, using the same example customer from notebook 04 (`CreditScore=600, Geography=France, Gender=Male, Age=40, Tenure=3, Balance=60000, NumOfProducts=2, HasCrCard=1, IsActiveMember=1, EstimatedSalary=50000`).

In [3]:
import pandas as pd

with open("label_encoder_gender.pkl", "rb") as file:
    label_encoder_gender = pickle.load(file)
with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

credit_score, gender, age, tenure, balance = 600, "Male", 40, 3, 60000
num_of_products, has_cr_card, is_active_member, estimated_salary, geography = 2, 1, 1, 50000, "France"

input_data = pd.DataFrame({
    "CreditScore": [credit_score],
    "Gender": [label_encoder_gender.transform([gender])[0]],
    "Age": [age],
    "Tenure": [tenure],
    "Balance": [balance],
    "NumOfProducts": [num_of_products],
    "HasCrCard": [has_cr_card],
    "IsActiveMember": [is_active_member],
    "EstimatedSalary": [estimated_salary],
})

geo_encoded = onehot_encoder_geo.transform([[geography]])  # already dense - the fixed version
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(["Geography"]))
input_data = pd.concat([input_data.reset_index(drop=True), geo_encoded_df], axis=1)

print("Column order matches scaler.feature_names_in_:", list(input_data.columns) == list(scaler.feature_names_in_))

input_data_scaled = scaler.transform(input_data)
prediction = model.predict(input_data_scaled, verbose=0)
prediction_proba = prediction[0][0]

print(f"Churn Probability: {prediction_proba:.4f}")
print("The customer is likely to churn." if prediction_proba > 0.5 else "The customer is not likely to churn.")

Column order matches scaler.feature_names_in_: True


C:\AI Projects\ML&DL\Learning\NLP Krish\Projects\ANN Classification\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


Churn Probability: 0.0338
The customer is not likely to churn.


Confirmed: **column order matches exactly**, and the final probability (**0.0338**) matches notebook 04's result on the same input — the rest of `app.py`'s logic is correct and consistent, once the `.toarray()` issue above is resolved.

## 7. Scaling, prediction, and displaying the result

```python
# Scale the input data
input_data_scaled = scaler.transform(input_data)

# Predict churn
prediction = model.predict(input_data_scaled)
prediction_proba = prediction[0][0]

st.write(f'Churn Probability: {prediction_proba:.2f}')

if prediction_proba > 0.5:
    st.write('The customer is likely to churn.')
else:
    st.write('The customer is not likely to churn.')
```

Same threshold logic as notebook 04 (`> 0.5`), just rendered with `st.write` instead of `print`.

## 8. Suggested improvements (not applied — your call whether to add them)

None of these are bugs, just worth knowing:

1. **Cache the model/encoder/scaler loading with `@st.cache_resource`.** As written, `app.py` reloads the model and all three pickles from disk on *every single* widget interaction — Streamlit re-runs the whole script top-to-bottom on every rerun. Model loading is the most expensive part of the script; wrapping it in a cached function means it only actually runs once per app session:
   ```python
   @st.cache_resource
   def load_artifacts():
       model = tf.keras.models.load_model('model.keras')
       with open('label_encoder_gender.pkl', 'rb') as f:
           label_encoder_gender = pickle.load(f)
       with open('onehot_encoder_geo.pkl', 'rb') as f:
           onehot_encoder_geo = pickle.load(f)
       with open('scaler.pkl', 'rb') as f:
           scaler = pickle.load(f)
       return model, label_encoder_gender, onehot_encoder_geo, scaler

   model, label_encoder_gender, onehot_encoder_geo, scaler = load_artifacts()
   ```
2. **`model.predict(input_data_scaled, verbose=0)`** — suppresses a progress-bar log line on every prediction; harmless either way, just a bit noisier in the terminal without it.
3. **Defensive column-order enforcement**, matching notebook 04's Section 5: `input_data = input_data[scaler.feature_names_in_]` right before scaling, so a future edit to the field-assembly order can't silently break predictions.

## 9. Running the app

```bash
streamlit run app.py
```

Opens a local web server (default `http://localhost:8501`) rendering the form; every widget change triggers a script re-run and a fresh prediction — exactly as demonstrated in the video (churn probability updating live as `Age`, `Geography`, `Balance`, etc. change).

## 10. Preparing for deployment: version control

Before pushing to GitHub (covered fully in the next notebook, once Streamlit Cloud deployment is shown), one practical addition worth making: a **`.gitignore`** file. None currently exists in this project, and without one, `git add .` would try to commit the entire `venv/` folder — hundreds of MB of environment-specific binaries that don't belong in version control (and would make cloning the repo painfully slow). A minimal `.gitignore` for this project would exclude at least `venv/`, `__pycache__/`, `*.pyc`, and `.ipynb_checkpoints/`. Let me know if you'd like me to create one.

## 11. Summary

- `app.py` mirrors notebook 04's prediction logic exactly, with Streamlit widgets replacing the hardcoded example input.
- Dropdown choices are populated directly from the fitted encoders (`categories_`, `classes_`) — the UI can never submit a value the model wasn't trained on.
- **Critical bug found and verified with real code:** the current `onehot_encoder_geo.pkl` was fit with `sparse_output=False`, but `app.py` calls `.toarray()` on its output anyway — this will crash the app the moment a prediction is attempted. Two fixes offered (re-fit the encoder with default sparse output, or drop `.toarray()` from `app.py`); neither applied yet, pending your choice.
- Everything else in the pipeline — column order, scaling, prediction — was independently re-verified against your real project files and confirmed correct (probability **0.0338** on the test customer, matching notebook 04 exactly).
- Three optional improvements suggested: `@st.cache_resource` for the expensive one-time loading, `verbose=0` on `predict`, and explicit column-order enforcement before scaling.
- No `.gitignore` exists yet — needed before committing to GitHub to avoid checking in the `venv/` folder.

## 12. Likely exam / interview questions

1. Why does a Streamlit script re-run top-to-bottom on every widget interaction, and what problem does that create for expensive setup code like model loading?
2. What does `@st.cache_resource` do, and why is it preferred over `@st.cache_data` for a loaded Keras model?
3. Why does `OneHotEncoder.transform()` sometimes return a sparse matrix and sometimes a dense array — what determines this, and when is it decided?
4. Why is populating dropdown options directly from `encoder.categories_` / `encoder.classes_` (rather than hardcoding them) good practice?
5. Why shouldn't a `venv/` folder be committed to a Git repository?

## 13. What's next

- Push the project to GitHub (with a proper `.gitignore`).
- Deploy the app to **Streamlit Cloud**.